# Допуск: Does Reinforcement Learning Really Incentivize Reasoning Capacity in LLMs Beyond the Base Model?

**Статья:** Y. Yue, Z. Chen, R. Lu, A. Zhao, Z. Wang, Y. Yue, S. Song, G. Huang, 2025 —
<https://arxiv.org/abs/2504.13837>

5 заданий. Под заголовком каждого задания указан его тип:

- **что есть что** — названия функций заменены: определите, что чему соответствует, и
  ответьте на вопросы;
- **расхождение со статьёй** — код реализует не то, что описано в статье: найдите это место;
- **неверный комментарий** — код корректен, но один комментарий утверждает неправду: найдите
  его;
- **вопрос по коду** — в коде нет отклонений: ответьте на вопросы, опираясь на код и статью.

Для найденного отклонения напишите: где оно, что не так, какое место статьи (раздел /
формула / таблица / цитата) нарушено и как должно быть. Код писать не нужно — достаточно
объяснить словами. Запускать и отлаживать код тоже не нужно: все задания решаются чтением
кода и статьи.

Общие импорты:

In [ ]:
import math, re
from collections import Counter

import torch

Во всех заданиях `correct` — булев тензор: `correct[i, j]` = «j-й сэмпл модели на i-й задаче
прошёл проверку». Функция `pass_at_k(correct, k)` (корректна, не показана) возвращает
средний по задачам pass@k, посчитанный так, как в статье. Функции, которые используются, но
не показаны, считайте корректными.

## Задание 1. Четыре способа оценить модель по n сэмплам

**Тип задания:** что есть что — названия функций заменены, определите, что чему соответствует.

На одну задачу у модели есть n ≥ k сэмплов. Функции ниже по первым k (или по всем n)
сэмплам выдают число от 0 до 1. Определите, какая функция — оценка pass@k, которую
используют авторы (§2.2, прил. A.2), какая — «наивная» оценка pass@k, какая — majority
voting, какая — Best-of-N. Ошибок в коде нет.

In [ ]:
def score_a(correct, k):                        # correct: (n,) bool, n >= k
    n, c = len(correct), int(correct.sum())
    if n - c < k:
        return 1.0
    return 1.0 - math.comb(n - c, k) / math.comb(n, k)

def score_b(correct, k):
    p = correct.float().mean().item()
    return 1.0 - (1.0 - p) ** k

def score_c(correct, answers, k):               # answers: list of n final answers
    top = Counter(answers[:k]).most_common(1)[0][0]
    return float(correct[answers.index(top)])

def score_d(correct, scores, k):                # scores: (n,) reward-model scores
    return float(correct[:k][scores[:k].argmax()])

**Вопросы.**
1. Пусть n = k = 256 и правильный ровно один сэмпл. Что вернёт каждая из четырёх функций
   (для `score_c` и `score_d` — в типичном случае)? Какой ответ соответствует определению
   pass@k из §2.2 и почему авторы для измерения *границы* рассуждений не берут majority
   voting и Best-of-N?
2. `score_b` выглядит естественно: «оценили вероятность успеха p и подставили в
   1 − (1 − p)^k». Почему авторы так не делают? В какую сторону ошибается эта оценка
   (завышает или занижает pass@k) и при каких k ошибка особенно велика?
3. Зачем вообще генерировать n > k сэмплов, если pass@k определён через k попыток
   (прил. A.2)?
4. При больших k на математике pass@k может засчитать задачу, где модель «угадала» ответ.
   Почему в задачах на генерацию кода (проверка юнит-тестами) эта проблема почти не
   возникает, и что авторы сделали для математики
   (§2.2, §3.1, прил. C.2)?

**Ваш ответ:**

_Напишите ответ здесь._

## Задание 2. Награда и шаг GRPO

**Тип задания:** расхождение со статьёй — код реализует не то, что описано в статье.

Кусок RLVR-обучения: проверка ответа, групповые advantage'и GRPO и policy-gradient loss
(`logp` — log π(y_t | x, y_<t) сэмплированных ответов). Найдите одно отклонение от статьи.

In [ ]:
def verify(response, gold):                     # r = V(x, y) in {0, 1}
    return float(gold in response)

def grpo_advantages(rewards):                   # rewards: (P, G), G responses per prompt
    mean = rewards.mean(1, keepdim=True)
    std = rewards.std(1, keepdim=True)
    return (rewards - mean) / (std + 1e-6)      # (P, G)

def policy_loss(logp, adv, mask):               # logp, mask: (P, G, T)
    return -(adv[..., None] * logp * mask).sum() / mask.sum()

**Ваш ответ:**

_Напишите ответ здесь._

## Задание 3. Сравнение base и RL-модели

**Тип задания:** расхождение со статьёй — код реализует не то, что описано в статье.

По 256 сэмплам на задачу для base- и RL-модели функция строит кривые pass@k, находит k,
начиная с которого base обгоняет RL, считает sampling efficiency gap и таблицу покрытия
задач. Найдите одно отклонение от статьи.

In [ ]:
def boundary_report(base_correct, rl_correct):
    # *_correct: (N, n) bool, n = 256 samples per problem
    ks = [2 ** i for i in range(9)]                               # 1 ... 256
    base = [pass_at_k(base_correct, k) for k in ks]
    rl = [pass_at_k(rl_correct, k) for k in ks]
    crossover = next((k for k, b, r in zip(ks, base, rl) if b > r), None)
    delta_se = pass_at_k(rl_correct, 256) - pass_at_k(rl_correct, 1)
    sb, sr = base_correct.any(1), rl_correct.any(1)               # (N,) solved at least once
    table = {
        "both": (sb & sr).float().mean().item(),
        "base only": (sb & ~sr).float().mean().item(),
        "rl only": (~sb & sr).float().mean().item(),
        "neither": (~sb & ~sr).float().mean().item(),
    }
    return base, rl, crossover, delta_se, table

**Ваш ответ:**

_Напишите ответ здесь._

## Задание 4. Perplexity ответов

**Тип задания:** неверный комментарий — код корректен, но один комментарий утверждает неправду.

`model` — causal LM из `transformers`, `tok` — её токенизатор. `y_base` и `y_rl` — списки
ответов, сэмплированных base- и RL-моделью на один и тот же `prompt`. Найдите один неверный
комментарий.

In [ ]:
@torch.no_grad()
def response_ppl(model, tok, prompt, response):
    p = tok(prompt, return_tensors="pt").input_ids
    r = tok(response, return_tensors="pt", add_special_tokens=False).input_ids
    ids = torch.cat([p, r], 1).to(model.device)                   # (1, P + T)
    logp = model(ids).logits[:, :-1].log_softmax(-1)              # (1, P + T - 1, V)
    tok_logp = logp.gather(-1, ids[:, 1:, None])[..., 0]          # (1, P + T - 1)
    return tok_logp[:, p.shape[1] - 1:].mean().neg().exp().item() # response tokens only

def compare(base_model, tok, prompt, y_base, y_rl):
    ppl_bb = [response_ppl(base_model, tok, prompt, y) for y in y_base]
    ppl_br = [response_ppl(base_model, tok, prompt, y) for y in y_rl]
    # If PPL_Base(Y_RL | x) sits in the low part of PPL_Base(Y_Base | x), the RL model
    # produces paths the base model would rarely sample: RLVR found new reasoning
    return ppl_bb, ppl_br

**Ваш ответ:**

_Напишите ответ здесь._

## Задание 5. Игрушечный RLVR

**Тип задания:** вопрос по коду — в коде нет отклонений, ответьте на вопросы.

Игрушечная модель эффекта RLVR. У каждой из N задач есть M «путей рассуждения»; base-модель
задана распределением над путями, часть путей ведёт к правильному ответу. «RL» усиливает
правильные пути, которые попались в G rollout'ах, и заостряет распределение. Для обеих
моделей берётся по n сэмплов на задачу и считается pass@k.

In [ ]:
def rlvr_like(logits, correct_path, G=8, beta=3.0, tau=0.5):
    # logits: (N, M) base log-probs over M reasoning paths, correct_path: (N, M) bool
    rollouts = torch.multinomial(logits.softmax(-1), G, replacement=True)    # (N, G)
    seen = torch.zeros_like(correct_path).scatter_(1, rollouts, True)        # (N, M)
    boosted = logits + beta * (seen & correct_path)       # reward only what was sampled
    return (boosted / tau).softmax(-1)                    # sharper policy, (N, M)

def simulate(N=500, M=200, n=256, **rl_kwargs):
    torch.manual_seed(0)
    logits = 2 * torch.randn(N, M)                        # base model over reasoning paths
    correct_path = torch.rand(N, M) < 0.03                # paths that end in a correct answer
    policies = {"base": logits.softmax(-1), "rl": rlvr_like(logits, correct_path, **rl_kwargs)}
    out = {}
    for name, pi in policies.items():
        p_ok = (pi * correct_path).sum(-1)                # (N,) P(correct sample)
        samples = torch.rand(N, n) < p_ok[:, None]        # (N, n) n samples per problem
        out[name] = {k: round(pass_at_k(samples, k), 3) for k in (1, 4, 16, 64, 256)}
    return out

**Вопросы.**
1. В этой модели «RL» выигрывает у base при k = 1 и проигрывает при k = 256 (как на рис. 2).
   Объясните по коду, за счёт каких задач растёт pass@1 и какие задачи «RL» перестаёт
   решать. Сопоставьте с задачами A и B на рис. 1 (слева) и с гистограммой на рис. 5.
2. Может ли здесь «RL» надёжно решать задачу, которую base не решает ни за какое разумное
   число сэмплов? Как это связано с табл. 2 и табл. 5?
3. Что в коде соответствует числу rollout'ов на промпт? Что, по-вашему, произойдёт с pass@256
   «RL», если его увеличить, и что об этом говорит статья (§4.4, рис. 16)?

**Ваш ответ:**

_Напишите ответ здесь._